# Enhanced Invoice Management Classifier Training
## 12-Action Model with Smart Entity Creation

This notebook trains a DistilBERT model for invoice management with enhanced entity creation capabilities.

**Action Classes (12 total):**
1. view_invoice - View specific invoice
2. list_invoices - List all invoices
3. create_invoice - Create basic invoice (form navigation)
4. edit_invoice - Edit existing invoice
5. list_customers - Show customers
6. list_products - Show products
7. show_reports - Show reports
8. overdue_invoices - Show overdue invoices
9. help - Show help
10. **create_product_with_data** - Create product with extracted data
11. **create_customer_with_data** - Create customer with extracted data
12. **create_invoice_with_data** - Create invoice with extracted data


In [ ]:
# Install required packages
!pip install transformers datasets torch scikit-learn numpy pandas matplotlib seaborn
!pip install optimum[onnxruntime]

In [ ]:
import torch
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from transformers import (
    DistilBertTokenizer, 
    DistilBertForSequenceClassification,
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from optimum.onnxruntime import ORTModelForSequenceClassification
from pathlib import Path

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"Using device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Enhanced Training Data with Entity Creation Commands

Comprehensive training dataset with ~150 examples per action class for balanced training.

In [ ]:
# Enhanced training data for 12 action classes
training_data = [
    # === EXISTING 9 ACTIONS (Enhanced with more variations) ===
    
    # view_invoice (150+ examples)
    "view invoice 123", "show invoice #456", "display invoice 789", "open invoice 101",
    "see invoice 202", "look at invoice 303", "check invoice 404", "invoice 505",
    "view invoice number 606", "show me invoice 707", "display invoice #808",
    "I want to see invoice 909", "can you show invoice 111", "pull up invoice 222",
    "open invoice #333", "view the invoice 444", "show invoice id 555",
    "display invoice with id 666", "look up invoice 777", "find invoice 888",
    "view invoice document 999", "show me the invoice 1001", "check invoice #1002",
    "see invoice number 1003", "open invoice document 1004", "view invoice 1005",
    "show invoice details 1006", "display invoice info 1007", "invoice details 1008",
    "view invoice #1009", "show invoice 1010", "open invoice 1011", "see invoice 1012",
    "view invoice 1013", "show invoice #1014", "display invoice 1015", "check invoice 1016",
    "look at invoice 1017", "see invoice 1018", "view invoice 1019", "show invoice 1020",
    "invoice 1021", "view invoice 1022", "show invoice #1023", "display invoice 1024",
    "open invoice 1025", "see invoice 1026", "view invoice 1027", "show invoice 1028",
    "check invoice 1029", "look at invoice 1030", "view invoice #1031", "show invoice 1032",
    "display invoice 1033", "open invoice 1034", "see invoice 1035", "view invoice 1036",
    "show invoice 1037", "invoice 1038", "view invoice #1039", "show invoice 1040",
    "display invoice 1041", "check invoice 1042", "look at invoice 1043", "see invoice 1044",
    "view invoice 1045", "show invoice #1046", "open invoice 1047", "display invoice 1048",
    "view invoice 1049", "show invoice 1050", "see invoice #1051", "check invoice 1052",
    "look at invoice 1053", "view invoice 1054", "show invoice #1055", "display invoice 1056",
    "open invoice 1057", "see invoice 1058", "view invoice 1059", "show invoice 1060",
    "invoice 1061", "view invoice #1062", "show invoice 1063", "display invoice 1064",
    "check invoice 1065", "look at invoice 1066", "see invoice 1067", "view invoice 1068",
    "show invoice #1069", "open invoice 1070", "display invoice 1071", "view invoice 1072",
    "show invoice 1073", "see invoice #1074", "check invoice 1075", "look at invoice 1076",
    "view invoice 1077", "show invoice #1078", "display invoice 1079", "open invoice 1080",
    "see invoice 1081", "view invoice 1082", "show invoice #1083", "invoice 1084",
    "display invoice 1085", "check invoice 1086", "look at invoice 1087", "view invoice 1088",
    "show invoice #1089", "open invoice 1090", "see invoice 1091", "display invoice 1092",
    "view invoice 1093", "show invoice #1094", "check invoice 1095", "look at invoice 1096",
    "see invoice 1097", "view invoice 1098", "show invoice #1099", "display invoice 1100",
    "open invoice 1101", "view invoice 1102", "show invoice #1103", "see invoice 1104",
    "check invoice 1105", "look at invoice 1106", "display invoice 1107", "view invoice 1108",
    "show invoice #1109", "open invoice 1110", "see invoice 1111", "view invoice 1112",
    "show invoice #1113", "display invoice 1114", "check invoice 1115", "look at invoice 1116",
    "view invoice 1117", "show invoice #1118", "open invoice 1119", "see invoice 1120",
    "display invoice 1121", "view invoice 1122", "show invoice #1123", "check invoice 1124",
    "look at invoice 1125", "see invoice 1126", "view invoice 1127", "show invoice #1128",
    "open invoice 1129", "display invoice 1130", "view invoice 1131", "show invoice #1132",
    "see invoice 1133", "check invoice 1134", "look at invoice 1135", "view invoice 1136",
    "show invoice #1137", "display invoice 1138", "open invoice 1139", "see invoice 1140",
    "view invoice 1141", "show invoice #1142", "check invoice 1143", "look at invoice 1144",
    "display invoice 1145", "view invoice 1146", "show invoice #1147", "open invoice 1148",
    "see invoice 1149", "view invoice 1150",
    
    # list_invoices (150+ examples)
    "list invoices", "show all invoices", "display invoices", "view invoices",
    "get all invoices", "show invoice list", "display all invoices", "see invoices",
    "list all invoices", "show me invoices", "get invoices", "view all invoices",
    "display invoice list", "show invoices", "list of invoices", "all invoices",
    "invoice list", "show me all invoices", "display all invoice data", "get invoice list",
    "view invoice list", "see all invoices", "show invoice data", "display invoices list",
    "list invoice data", "get all invoice info", "show all invoice records", "view invoice records",
    "display invoice records", "see invoice list", "show invoice table", "view invoice table",
    "display invoice table", "get invoice table", "list invoice table", "show me invoice table",
    "see invoice records", "view all invoice data", "display all invoice records", "show invoice info",
    "get invoice info", "list invoice info", "view invoice info", "see invoice info",
    "display invoice info", "show all invoice details", "get invoice details", "list invoice details",
    "view invoice details", "see invoice details", "display invoice details", "show invoice overview",
    "get invoice overview", "list invoice overview", "view invoice overview", "see invoice overview",
    "display invoice overview", "invoices", "all invoice data", "invoice data",
    "invoice records", "invoice table", "invoice details", "invoice overview",
    "show invoices list", "display invoices data", "get invoices list", "view invoices data",
    "see invoices list", "list invoices data", "show all invoice items", "display all invoice items",
    "get all invoice items", "view all invoice items", "see all invoice items", "list all invoice items",
    "show invoice items", "display invoice items", "get invoice items", "view invoice items",
    "see invoice items", "list invoice items", "invoice items", "all invoice items",
    "show me invoices data", "display invoices table", "get invoices table", "view invoices table",
    "see invoices table", "list invoices table", "invoices table", "invoices data",
    "invoices records", "invoices list", "invoices info", "invoices details",
    "invoices overview", "invoices items", "show all invoices data", "display all invoices table",
    "get all invoices table", "view all invoices table", "see all invoices table", "list all invoices table",
    "all invoices table", "all invoices data", "all invoices records", "all invoices list",
    "all invoices info", "all invoices details", "all invoices overview", "all invoices items",
    "show me all invoices data", "display all invoices records", "get all invoices records", "view all invoices records",
    "see all invoices records", "list all invoices records", "show all invoices list", "display all invoices list",
    "get all invoices list", "view all invoices list", "see all invoices list", "list all invoices list",
    "show all invoices info", "display all invoices info", "get all invoices info", "view all invoices info",
    "see all invoices info", "list all invoices info", "show all invoices details", "display all invoices details",
    "get all invoices details", "view all invoices details", "see all invoices details", "list all invoices details",
    "show all invoices overview", "display all invoices overview", "get all invoices overview", "view all invoices overview",
    "see all invoices overview", "list all invoices overview", "show all invoices items", "display all invoices items",
    "get all invoices items", "view all invoices items", "see all invoices items", "list all invoices items",
    "show invoices summary", "display invoices summary", "get invoices summary", "view invoices summary",
    "see invoices summary", "list invoices summary", "invoices summary", "all invoices summary",
    "show all invoices summary", "display all invoices summary", "get all invoices summary", "view all invoices summary",
    "see all invoices summary", "list all invoices summary", "show invoice summary", "display invoice summary",
    "get invoice summary", "view invoice summary", "see invoice summary", "list invoice summary",
    "invoice summary", "all invoice summary", "show all invoice summary", "display all invoice summary",
    "get all invoice summary", "view all invoice summary", "see all invoice summary", "list all invoice summary",
    
    # create_invoice (50+ examples)
    "create invoice", "new invoice", "add invoice", "create new invoice",
    "make invoice", "generate invoice", "add new invoice", "create an invoice",
    "make new invoice", "generate new invoice", "start new invoice", "begin invoice",
    "create invoice form", "new invoice form", "add invoice form", "make invoice form",
    "generate invoice form", "start invoice form", "begin invoice form", "invoice creation",
    "new invoice creation", "invoice form", "create invoice document", "new invoice document",
    "add invoice document", "make invoice document", "generate invoice document", "start invoice document",
    "begin invoice document", "invoice document creation", "new invoice document creation", "invoice document form",
    "create invoice entry", "new invoice entry", "add invoice entry", "make invoice entry",
    "generate invoice entry", "start invoice entry", "begin invoice entry", "invoice entry creation",
    "new invoice entry creation", "invoice entry form", "create invoice record", "new invoice record",
    "add invoice record", "make invoice record", "generate invoice record", "start invoice record",
    "begin invoice record", "invoice record creation", "new invoice record creation", "invoice record form",
    
    # edit_invoice (50+ examples)
    "edit invoice 123", "modify invoice 456", "update invoice 789", "change invoice 101",
    "edit invoice #202", "modify invoice #303", "update invoice #404", "change invoice #505",
    "edit invoice number 606", "modify invoice number 707", "update invoice number 808", "change invoice number 909",
    "edit the invoice 111", "modify the invoice 222", "update the invoice 333", "change the invoice 444",
    "edit invoice id 555", "modify invoice id 666", "update invoice id 777", "change invoice id 888",
    "edit invoice with id 999", "modify invoice with id 1001", "update invoice with id 1002", "change invoice with id 1003",
    "edit invoice document 1004", "modify invoice document 1005", "update invoice document 1006", "change invoice document 1007",
    "edit invoice entry 1008", "modify invoice entry 1009", "update invoice entry 1010", "change invoice entry 1011",
    "edit invoice record 1012", "modify invoice record 1013", "update invoice record 1014", "change invoice record 1015",
    "edit invoice details 1016", "modify invoice details 1017", "update invoice details 1018", "change invoice details 1019",
    "edit invoice info 1020", "modify invoice info 1021", "update invoice info 1022", "change invoice info 1023",
    "edit invoice data 1024", "modify invoice data 1025", "update invoice data 1026", "change invoice data 1027",
    "edit invoice form 1028", "modify invoice form 1029", "update invoice form 1030", "change invoice form 1031",
    
    # list_customers (50+ examples)
    "list customers", "show customers", "display customers", "view customers",
    "get customers", "show all customers", "display all customers", "view all customers",
    "get all customers", "list all customers", "customer list", "customers",
    "all customers", "show customer list", "display customer list", "view customer list",
    "get customer list", "customer data", "all customer data", "show customer data",
    "display customer data", "view customer data", "get customer data", "list customer data",
    "customer records", "all customer records", "show customer records", "display customer records",
    "view customer records", "get customer records", "list customer records", "customer table",
    "all customer table", "show customer table", "display customer table", "view customer table",
    "get customer table", "list customer table", "customer info", "all customer info",
    "show customer info", "display customer info", "view customer info", "get customer info",
    "list customer info", "customer details", "all customer details", "show customer details",
    "display customer details", "view customer details", "get customer details", "list customer details",
    
    # list_products (50+ examples)
    "list products", "show products", "display products", "view products",
    "get products", "show all products", "display all products", "view all products",
    "get all products", "list all products", "product list", "products",
    "all products", "show product list", "display product list", "view product list",
    "get product list", "product data", "all product data", "show product data",
    "display product data", "view product data", "get product data", "list product data",
    "product records", "all product records", "show product records", "display product records",
    "view product records", "get product records", "list product records", "product table",
    "all product table", "show product table", "display product table", "view product table",
    "get product table", "list product table", "product info", "all product info",
    "show product info", "display product info", "view product info", "get product info",
    "list product info", "product details", "all product details", "show product details",
    "display product details", "view product details", "get product details", "list product details",
    
    # show_reports (50+ examples)
    "show reports", "display reports", "view reports", "get reports",
    "list reports", "reports", "all reports", "show all reports",
    "display all reports", "view all reports", "get all reports", "list all reports",
    "report data", "all report data", "show report data", "display report data",
    "view report data", "get report data", "list report data", "reporting",
    "show reporting", "display reporting", "view reporting", "get reporting",
    "list reporting", "report dashboard", "show report dashboard", "display report dashboard",
    "view report dashboard", "get report dashboard", "list report dashboard", "dashboard",
    "show dashboard", "display dashboard", "view dashboard", "get dashboard",
    "list dashboard", "analytics", "show analytics", "display analytics",
    "view analytics", "get analytics", "list analytics", "report summary",
    "show report summary", "display report summary", "view report summary", "get report summary",
    "list report summary", "summary", "show summary", "display summary",
    "view summary", "get summary", "list summary", "statistics",
    
    # overdue_invoices (50+ examples)
    "overdue invoices", "show overdue invoices", "display overdue invoices", "view overdue invoices",
    "get overdue invoices", "list overdue invoices", "overdue", "show overdue",
    "display overdue", "view overdue", "get overdue", "list overdue",
    "late invoices", "show late invoices", "display late invoices", "view late invoices",
    "get late invoices", "list late invoices", "late payments", "show late payments",
    "display late payments", "view late payments", "get late payments", "list late payments",
    "past due invoices", "show past due invoices", "display past due invoices", "view past due invoices",
    "get past due invoices", "list past due invoices", "past due", "show past due",
    "display past due", "view past due", "get past due", "list past due",
    "unpaid invoices", "show unpaid invoices", "display unpaid invoices", "view unpaid invoices",
    "get unpaid invoices", "list unpaid invoices", "unpaid", "show unpaid",
    "display unpaid", "view unpaid", "get unpaid", "list unpaid",
    "outstanding invoices", "show outstanding invoices", "display outstanding invoices", "view outstanding invoices",
    "get outstanding invoices", "list outstanding invoices", "outstanding", "show outstanding",
    
    # help (50+ examples)
    "help", "help me", "assistance", "support", "guide", "instructions",
    "how to use", "what can you do", "commands", "features",
    "show help", "display help", "view help", "get help", "list help",
    "help menu", "show help menu", "display help menu", "view help menu", "get help menu",
    "list help menu", "help options", "show help options", "display help options", "view help options",
    "get help options", "list help options", "help commands", "show help commands", "display help commands",
    "view help commands", "get help commands", "list help commands", "available commands", "show available commands",
    "display available commands", "view available commands", "get available commands", "list available commands", "what commands",
    "show what commands", "display what commands", "view what commands", "get what commands", "list what commands",
    "how do I", "show how do I", "display how do I", "view how do I",
    "get how do I", "list how do I", "tutorial", "show tutorial", "display tutorial",
    "view tutorial", "get tutorial", "list tutorial", "documentation", "show documentation",
    
    # === NEW ENTITY CREATION ACTIONS (150+ examples each) ===
    
    # create_product_with_data (150+ examples)
    "create product WaterBottle price 5 dollars",
    "add product Laptop description gaming laptop price 1200",
    "make product Coffee Mug priced at 15 USD",
    "create new product Smartphone description iPhone 14 cost 999",
    "add new product Headphones with price 200 dollars",
    "make new product Keyboard description mechanical keyboard price 150",
    "generate product Mouse with price 50",
    "create product Monitor description 4K monitor priced at 500",
    "add product Webcam price 100 dollars",
    "make product Speaker description Bluetooth speaker cost 80",
    "create product Tablet with description iPad Air price 600",
    "add product Printer description laser printer priced at 300",
    "make product Scanner with price 250 dollars",
    "generate product Router description wireless router cost 120",
    "create product Cable description USB-C cable price 25",
    "add product Charger with description fast charger priced at 40",
    "make product Case description phone case cost 30",
    "create product Stand with price 75 dollars",
    "add product Dock description laptop dock priced at 200",
    "make product Bag with description laptop bag cost 60",
    "create product WaterBottle which is a product description and its price is 5 dollars",
    "add product named Laptop with description gaming laptop and price 1200",
    "make a product Coffee Mug that costs 15 USD",
    "create new product called Smartphone iPhone 14 for 999 dollars",
    "add a new product Headphones priced at 200",
    "make new product Keyboard mechanical type price 150",
    "generate a product Mouse costing 50 dollars",
    "create product Monitor 4K display for 500",
    "add product Webcam at 100 dollars",
    "make product Speaker Bluetooth type costs 80",
    "create product Book title Python Programming price 35",
    "add product Pen description ballpoint pen cost 2",
    "make product Notebook spiral bound price 8",
    "create product Calculator scientific type costs 45",
    "add product Ruler plastic 12 inch price 3",
    "make product Eraser white type costs 1",
    "create product Pencil mechanical price 12",
    "add product Marker permanent black costs 4",
    "make product Highlighter yellow price 6",
    "create product Stapler metal type costs 18",
    "add product Paperclips box of 100 price 5",
    "make product Folder manila type costs 2",
    "create product Binder 3 ring price 12",
    "add product Paper white 500 sheets costs 8",
    "make product Envelope white letter size price 10",
    "create product Stamp book of 20 costs 15",
    "add product Label sheet of 30 price 6",
    "make product Tape clear 1 inch costs 4",
    "create product Scissors 8 inch price 10",
    "add product Glue stick white costs 3",
    "make product Rubber band box of 100 price 4",
    "create product Pushpin box of 50 costs 3",
    "add product Thumbtack brass type price 5",
    "make product Clip binder large costs 2",
    "create product Fastener brass type price 6",
    "add product Divider plastic 5 tab costs 8",
    "make product Index card 3x5 pack price 4",
    "create product Sticky note yellow 3x3 costs 3",
    "add product Calendar wall 2024 price 12",
    "make product Planner daily type costs 25",
    "create product Whiteboard 24x36 price 45",
    "add product Marker board dry erase costs 35",
    "make product Easel tabletop price 30",
    "create product Flipchart pad 25 sheets costs 15",
    "add product Presentation remote wireless price 40",
    "make product Projector portable costs 300",
    "create product Screen projection 80 inch price 150",
    "add product Pointer laser red costs 20",
    "make product Timer digital price 15",
    "create product Clock wall analog costs 25",
    "add product Watch digital price 50",
    "make product Alarm clock battery costs 18",
    "create product Radio portable FM price 35",
    "add product Speaker portable Bluetooth costs 60",
    "make product Microphone USB type price 80",
    "create product Camera digital compact costs 200",
    "add product Lens camera 50mm price 300",
    "make product Tripod camera aluminum costs 120",
    "create product Bag camera shoulder price 45",
    "add product Strap camera neck costs 15",
    "make product Card memory 64GB price 25",
    "create product Battery rechargeable AA costs 12",
    "add product Charger battery USB price 20",
    "make product Adapter power USB costs 15",
    "create product Hub USB 4 port price 30",
    "add product Switch ethernet 8 port costs 50",
    "make product Modem cable high speed price 80",
    "create product Antenna wifi external costs 25",
    "add product Extender wifi range price 60",
    "make product Booster signal cell costs 100",
    "create product Phone cordless DECT price 45",
    "add product Headset wireless Bluetooth costs 120",
    "make product Earbuds noise canceling price 180",
    "create product Amplifier audio 2 channel costs 200",
    "add product Equalizer graphic 10 band price 150",
    "make product Mixer audio 4 channel costs 250",
    "create product Recorder digital voice price 75",
    "add product Player MP3 portable costs 85",
    "make product Radio internet wifi price 120",
    "create product Tuner FM digital costs 95",
    "add product Antenna radio telescoping price 35",
    "make product Receiver satellite costs 180",
    "create product Dish satellite 18 inch price 60",
    "add product Mount antenna roof costs 40",
    "make product Cable coaxial 50 foot price 25",
    "create product Connector BNC male costs 5",
    "add product Splitter signal 2 way price 8",
    "make product Amplifier signal inline costs 15",
    "create product Filter noise RF costs 12",
    "add product Attenuator signal 10dB price 10",
    "make product Terminator 75 ohm costs 3",
    "create product Barrel connector female price 2",
    "add product Ground block coax costs 6",
    "make product Weather boot rubber price 4",
    "create product Sealant cable entry costs 8",
    "add product Conduit flexible 1 inch price 12",
    "make product Fitting conduit coupling costs 3",
    "create product Clamp cable strain relief price 2",
    "add product Tie cable plastic costs 5",
    "make product Sleeve cable protective price 8",
    "create product Grommet rubber 1 inch costs 1",
    "add product Bushing plastic cable price 2",
    "make product Guard cable metal costs 10",
    "create product Cover cable decorative price 15",
    "add product Raceway cable surface costs 20",
    "make product Duct cable underground price 25",
    "create product Vault cable concrete costs 100",
    "add product Pedestal cable green price 150",
    "make product Enclosure cable weatherproof costs 75",
    "create product Box junction plastic price 8",
    "add product Panel patch 24 port costs 45",
    "make product Jack keystone Cat6 price 3",
    "create product Plate wall 2 port costs 5",
    "add product Frame patch rack costs 200",
    "make product Shelf rack 19 inch price 35",
    "create product Rail rack mounting costs 15",
    "add product Screw rack 10-32 price 1",
    "make product Nut rack cage costs 2",
    "create product Washer rack split price 1",
    "add product Standoff rack spacer costs 3",
    "make product Bracket rack L shape price 8",
    "create product Support rack vertical costs 25",
    "add product Guide cable vertical price 30",
    "make product Manager cable horizontal costs 40",
    "create product Tray cable sliding price 50",
    "add product Finger cable duct costs 20",
    "make product Ring cable D shape price 5",
    "create product Hook cable J shape costs 3",
    "add product Bar cable support price 15",
    "make product Ladder cable overhead costs 60",
    "create product Trough cable under floor price 45",
    "add product Tile access floor costs 25",
    "make product Grate floor ventilation price 30",
    "create product Fan cooling 120mm costs 20",
    "add product Filter fan washable price 8",
    "make product Guard fan metal costs 5",
    "create product Mount fan vibration price 10",
    "add product Controller fan speed costs 25",
    "make product Sensor temperature digital price 15",
    "create product Probe humidity analog costs 30",
    "add product Monitor environmental costs 200",
    "make product Alarm temperature audible price 50",
    "create product Light indicator LED costs 8",
    "add product Switch toggle SPDT price 5",
    "make product Button push momentary costs 3",
    "create product Relay DPDT 12V costs 8",
    "add product Socket relay 8 pin price 2",
    "make product Base relay DIN rail costs 5",
    "create product Terminal screw 10A costs 3",
    "add product Block terminal 12 position price 15",
    "make product Strip terminal DIN costs 25",
    "create product Marker terminal number costs 5",
    "add product End terminal barrier price 2",
    "make product Jumper terminal short costs 1",
    "create product Bridge terminal 10 way price 8",

    # create_customer_with_data (150+ examples)
    "add customer John Smith email john@example.com phone 555-1234",
    "create customer Sarah Johnson address 123 Main St phone 555-5678",
    "register customer Mike Wilson email mike@test.com address 456 Oak Ave",
    "add new customer Lisa Brown phone 555-9012 email lisa@company.com",
    "create new customer David Lee address 789 Pine St email david@email.com",
    "make customer Jennifer Davis phone 555-3456 address 321 Elm St",
    "add customer Robert Taylor email rob@domain.com phone 555-7890",
    "create customer Amanda White address 654 Maple Ave phone 555-2345",
    "register customer Chris Anderson email chris@site.com address 987 Cedar St",
    "add new customer Michelle Garcia phone 555-6789 email michelle@web.com",
    "create new customer James Martinez address 147 Birch Ln email james@mail.com",
    "make customer Patricia Rodriguez phone 555-1357 address 258 Ash St",
    "add customer William Hernandez email will@example.org phone 555-2468",
    "create customer Mary Lopez address 369 Spruce Ave phone 555-8024",
    "register customer Charles Gonzalez email charles@test.org address 741 Fir St",
    "add new customer Elizabeth Perez phone 555-9753 email liz@company.org",
    "create new customer Thomas Turner address 852 Willow Ln email tom@email.org",
    "make customer Maria Phillips phone 555-1470 address 963 Poplar St",
    "add customer Joseph Campbell email joe@domain.org phone 555-7410",
    "create customer Susan Evans address 159 Hickory Ave phone 555-8520",
    "add customer named John Smith with email john@example.com and phone 555-1234",
    "create a customer Sarah Johnson living at 123 Main St with phone 555-5678",
    "register new customer Mike Wilson email mike@test.com at address 456 Oak Ave",
    "add a new customer Lisa Brown phone number 555-9012 email lisa@company.com",
    "create customer David Lee at 789 Pine St with email david@email.com",
    "make a customer Jennifer Davis phone 555-3456 living at 321 Elm St",
    "add customer Robert Taylor with email rob@domain.com phone 555-7890",
    "create customer Amanda White living at 654 Maple Ave phone 555-2345",
    "register customer Chris Anderson email chris@site.com at 987 Cedar St",
    "add new customer Michelle Garcia with phone 555-6789 email michelle@web.com",
    "create customer named Alice Cooper email alice@music.com phone 555-ROCK",
    "add customer Bob Dylan address 61 Highway phone 555-FOLK",
    "make customer Elvis Presley email king@graceland.com address Memphis TN",
    "create customer Madonna phone 555-VOGUE email madonna@pop.com",
    "add customer Prince address Purple Rain St email prince@music.com",
    "register customer Michael Jackson phone 555-BEAT email mj@thriller.com",
    "create customer Whitney Houston address I Will Always Love You Lane",
    "add customer Aretha Franklin email respect@soul.com phone 555-SOUL",
    "make customer Stevie Wonder phone 555-MUSIC email stevie@motown.com",
    "create customer Ray Charles address Georgia On My Mind St",
    "add customer Ella Fitzgerald email ella@jazz.com phone 555-JAZZ",
    "register customer Frank Sinatra phone 555-BLUE email frank@vegas.com",
    "create customer Dean Martin address That's Amore Ave phone 555-DEAN",
    "add customer Sammy Davis Jr email sammy@ratpack.com address Vegas Strip",
    "make customer Tony Bennett phone 555-HEART email tony@sanfran.com",
    "create customer Nat King Cole address Unforgettable St",
    "add customer Bing Crosby email bing@christmas.com phone 555-XMAS",
    "register customer Louis Armstrong phone 555-JAZZ email louis@neworleans.com",
    "create customer Duke Ellington address Sophisticated Lady Ave",
    "add customer Count Basie email count@swing.com phone 555-SWING",
    "make customer Benny Goodman phone 555-CLARINET address Swing Era St",
    "create customer Glenn Miller email glenn@big.band phone 555-BAND",
    "add customer Tommy Dorsey address Trombone Ave phone 555-TOMMY",
    "register customer Artie Shaw email artie@clarinet.com address Begin Beguine St",
    "create customer Harry James phone 555-TRUMPET email harry@big.band",
    "add customer Les Brown address Sentimental Journey Ave",
    "make customer Woody Herman phone 555-HERD email woody@progressive.com",
    "create customer Stan Kenton address Innovation St phone 555-PROGRESSIVE",
    "add customer Maynard Ferguson email maynard@trumpet.com phone 555-HIGH",
    "register customer Buddy Rich address Drum Solo St phone 555-DRUMS",
    "create customer Gene Krupa email gene@drums.com address Sing Sing Sing Ave",
    "add customer Art Tatum phone 555-PIANO email art@virtuoso.com",
    "make customer Oscar Peterson address Canada Jazz St phone 555-OSCAR",
    "create customer Bill Evans email bill@piano.trio phone 555-WALTZ",
    "add customer Thelonious Monk address Round Midnight Ave phone 555-MONK",
    "register customer Miles Davis email miles@trumpet.com phone 555-MILES",
    "create customer John Coltrane address Giant Steps St phone 555-TRANE",
    "add customer Charlie Parker email bird@bebop.com phone 555-BIRD",
    "make customer Dizzy Gillespie phone 555-DIZZY address Salt Peanuts Ave",
    "create customer Clifford Brown email clifford@trumpet.com phone 555-BROWNIE",
    "add customer Lee Morgan address Sidewinder St phone 555-LEE",
    "register customer Freddie Hubbard email freddie@trumpet.com phone 555-FREDDIE",
    "create customer Wynton Marsalis address Lincoln Center phone 555-WYNTON",
    "add customer Branford Marsalis email branford@sax.com phone 555-BRANFORD",
    "make customer Herbie Hancock phone 555-HERBIE address Watermelon Man St",
    "create customer Chick Corea email chick@electric.piano phone 555-CHICK",
    "add customer Keith Jarrett address Koeln Concert Ave phone 555-KEITH",
    "register customer McCoy Tyner email mccoy@piano.com phone 555-MCCOY",
    "create customer Cecil Taylor address Unit Structures St phone 555-CECIL",
    "add customer Sun Ra email sun@saturn.com phone 555-SPACE",
    "make customer John Zorn phone 555-NAKED address Downtown NYC",
    "create customer Pat Metheny email pat@bright.size phone 555-PAT",
    "add customer Jaco Pastorius address Portrait of Tracy St phone 555-JACO",
    "register customer Stanley Clarke email stanley@bass.com phone 555-STANLEY",
    "create customer Marcus Miller phone 555-MARCUS email marcus@bass.com",
    "add customer Victor Wooten address Bass Extremes Ave phone 555-VICTOR",
    "make customer Bootsy Collins phone 555-FUNK email bootsy@pfunk.com",
    "create customer Larry Graham address One Nation Under Groove St",
    "add customer Flea email flea@chili.peppers phone 555-FLEA",
    "register customer Les Claypool phone 555-PRIMUS email les@south.park",
    "create customer Geddy Lee address Rush Ave phone 555-YYZ",
    "add customer Chris Squire email chris@yes.com phone 555-YES",
    "make customer John Entwistle phone 555-WHO address Ox St",
    "create customer Paul McCartney email paul@beatles.com address Abbey Road",
    "add customer John Lennon phone 555-IMAGINE email john@strawberry.fields",
    "register customer George Harrison email george@my.sweet phone 555-GUITAR",
    "create customer Ringo Starr address Yellow Submarine St phone 555-RINGO",
    "add customer Mick Jagger email mick@satisfaction.com phone 555-STONES",
    "make customer Keith Richards phone 555-KEEF address Exile Main St",
    "create customer Charlie Watts email charlie@drums.com address Steady Beat Ave",
    "add customer Ron Wood phone 555-WOODY email ron@faces.com",
    "register customer Jimmy Page email jimmy@zeppelin.com phone 555-PAGE",
    "create customer Robert Plant address Stairway to Heaven St phone 555-PLANT",
    "add customer John Paul Jones email jpj@bass.keys phone 555-JPJ",
    "make customer John Bonham phone 555-BONZO address When Levee Breaks Ave",
    "create customer Roger Waters email roger@pink.floyd phone 555-WATERS",
    "add customer David Gilmour address Comfortably Numb St phone 555-DAVID",
    "register customer Nick Mason email nick@drums.floyd phone 555-NICK",
    "create customer Richard Wright address Great Gig Sky phone 555-RICK",
    "add customer Syd Barrett email syd@diamond.dogs phone 555-SYD",

    # create_invoice_with_data (150+ examples)
    "create invoice for customer John with product Laptop quantity 2 at 1000 each",
    "make invoice for Sarah 5 Coffee Mugs at 15 dollars each",
    "new invoice customer Mike product WaterBottle qty 10 price 5",
    "generate invoice for Lisa with 3 Smartphones at 999 per unit",
    "add invoice customer David 1 Monitor at 500 dollars",
    "create invoice for Jennifer with 2 Headphones at 200 each",
    "make invoice customer Robert 4 Keyboards at 150 per piece",
    "new invoice for Amanda with 1 Printer at 300 dollars",
    "generate invoice customer Chris 6 Mice at 50 each",
    "add invoice for Michelle with 2 Tablets at 600 per unit",
    "create invoice customer James 1 Speaker at 80 dollars",
    "make invoice for Patricia with 3 Webcams at 100 each",
    "new invoice customer William 2 Routers at 120 per piece",
    "generate invoice for Mary with 5 Cables at 25 each",
    "add invoice customer Charles 1 Scanner at 250 dollars",
    "create invoice for Elizabeth with 4 Chargers at 40 per unit",
    "make invoice customer Thomas 2 Cases at 30 each",
    "new invoice for Maria with 1 Dock at 200 dollars",
    "generate invoice customer Joseph 3 Stands at 75 per piece",
    "add invoice for Susan with 2 Bags at 60 each",
    "create invoice for customer John Smith with product Laptop quantity 2 at 1000 dollars each",
    "make an invoice for Sarah Johnson 5 Coffee Mugs at 15 USD per item",
    "new invoice for customer Mike Wilson product WaterBottle quantity 10 price 5 each",
    "generate an invoice for Lisa Brown with 3 Smartphones at 999 per unit",
    "add invoice for customer David Lee 1 Monitor at 500 dollars",
    "create invoice for Jennifer Davis with 2 Headphones at 200 each",
    "make invoice for customer Robert Taylor 4 Keyboards at 150 per piece",
    "new invoice for Amanda White with 1 Printer at 300 dollars",
    "generate invoice for customer Chris Anderson 6 Mice at 50 each",
    "add invoice for Michelle Garcia with 2 Tablets at 600 per unit",
    "create invoice customer Alice Cooper 1 Guitar at 2000 dollars",
    "make invoice for Bob Dylan 1 Harmonica at 50 dollars",
    "new invoice customer Elvis Presley 1 Microphone at 300 dollars",
    "generate invoice for Madonna 1 Stage Light at 500 dollars",
    "add invoice customer Prince 1 Purple Piano at 5000 dollars",
    "create invoice for Michael Jackson 1 Sequined Glove at 10000 dollars",
    "make invoice customer Whitney Houston 1 Vocal Coach Session at 200 dollars",
    "new invoice for Aretha Franklin 1 Church Organ at 15000 dollars",
    "generate invoice customer Stevie Wonder 1 Synthesizer at 3000 dollars",
    "add invoice for Ray Charles 1 Piano Lesson at 100 dollars",
    "create invoice customer Ella Fitzgerald 1 Jazz Vocal Book at 25 dollars",
    "make invoice for Frank Sinatra 1 Tuxedo at 800 dollars",
    "new invoice customer Dean Martin 1 Martini Glass Set at 150 dollars",
    "generate invoice for Sammy Davis Jr 1 Tap Shoes at 200 dollars",
    "add invoice customer Tony Bennett 1 Heart Pendant at 500 dollars",
    "create invoice for Nat King Cole 1 Unforgettable Album at 30 dollars",
    "make invoice customer Bing Crosby 1 Christmas Album at 25 dollars",
    "new invoice for Louis Armstrong 1 Trumpet at 1200 dollars",
    "generate invoice customer Duke Ellington 1 Orchestra Arrangement at 500 dollars",
    "add invoice for Count Basie 1 Swing Dance Lesson at 75 dollars",
    "create invoice customer Benny Goodman 1 Clarinet at 800 dollars",
    "make invoice for Glenn Miller 1 Big Band Chart at 100 dollars",
    "new invoice customer Tommy Dorsey 1 Trombone at 900 dollars",
    "generate invoice for Artie Shaw 1 Begin the Beguine Sheet Music at 20 dollars",
    "add invoice customer Harry James 1 Trumpet Lesson at 80 dollars",
    "create invoice for Les Brown 1 Sentimental Journey CD at 15 dollars",
    "make invoice customer Woody Herman 1 Progressive Jazz Book at 35 dollars",
    "new invoice for Stan Kenton 1 Innovation Orchestra Score at 200 dollars",
    "generate invoice customer Maynard Ferguson 1 High Note Trumpet at 1500 dollars",
    "add invoice for Buddy Rich 1 Drum Solo Video at 40 dollars",
    "create invoice customer Gene Krupa 1 Drum Set at 2500 dollars",
    "make invoice for Art Tatum 1 Piano Virtuoso Masterclass at 300 dollars",
    "new invoice customer Oscar Peterson 1 Canadian Jazz History Book at 45 dollars",
    "generate invoice for Bill Evans 1 Waltz for Debby Album at 25 dollars",
    "add invoice customer Thelonious Monk 1 Round Midnight Sheet Music at 30 dollars",
    "create invoice for Miles Davis 1 Trumpet Mute at 150 dollars",
    "make invoice customer John Coltrane 1 Giant Steps Transcription at 50 dollars",
    "new invoice for Charlie Parker 1 Bebop Saxophone Book at 40 dollars",
    "generate invoice customer Dizzy Gillespie 1 Salt Peanuts Recording at 20 dollars",
    "add invoice for Clifford Brown 1 Trumpet Method Book at 35 dollars",
    "create invoice customer Lee Morgan 1 Sidewinder Album at 25 dollars",
    "make invoice for Freddie Hubbard 1 Jazz Trumpet Lesson at 100 dollars",
    "new invoice customer Wynton Marsalis 1 Lincoln Center Concert Ticket at 75 dollars",
    "generate invoice for Branford Marsalis 1 Saxophone Reed Box at 30 dollars",
    "add invoice customer Herbie Hancock 1 Watermelon Man CD at 18 dollars",
    "create invoice for Chick Corea 1 Electric Piano at 4000 dollars",
    "make invoice customer Keith Jarrett 1 Koeln Concert Vinyl at 35 dollars",
    "new invoice for McCoy Tyner 1 Piano Masterclass at 250 dollars",
    "generate invoice customer Cecil Taylor 1 Unit Structures Score at 100 dollars",
    "add invoice for Sun Ra 1 Saturn Keyboard at 3000 dollars",
    "create invoice customer John Zorn 1 Naked City Album at 22 dollars",
    "make invoice for Pat Metheny 1 Bright Size Life Guitar at 2500 dollars",
    "new invoice customer Jaco Pastorius 1 Fretless Bass at 3500 dollars",
    "generate invoice for Stanley Clarke 1 Bass Guitar Lesson at 120 dollars",
    "add invoice customer Marcus Miller 1 Bass Amplifier at 1800 dollars",
    "create invoice for Victor Wooten 1 Bass Extremes DVD at 30 dollars",
    "make invoice customer Bootsy Collins 1 Funk Bass Course at 150 dollars",
    "new invoice for Larry Graham 1 Slap Bass Tutorial at 80 dollars",
    "generate invoice customer Flea 1 Chili Peppers Bass Tab Book at 25 dollars",
    "add invoice for Les Claypool 1 Primus Bass Lesson at 100 dollars",
    "create invoice customer Geddy Lee 1 Rush Bass Collection at 40 dollars",
    "make invoice for Chris Squire 1 Yes Bass Parts Book at 35 dollars",
    "new invoice customer John Entwistle 1 Who Bass Anthology at 30 dollars",
    "generate invoice for Paul McCartney 1 Beatles Bass Book at 28 dollars",
    "add invoice customer John Lennon 1 Imagine Piano Score at 22 dollars",
    "create invoice for George Harrison 1 My Sweet Lord Guitar Tab at 18 dollars",
    "make invoice customer Ringo Starr 1 Yellow Submarine Drum Kit at 3000 dollars",
    "new invoice for Mick Jagger 1 Satisfaction Vocal Lesson at 200 dollars",
    "generate invoice customer Keith Richards 1 Exile Main St Guitar at 4000 dollars",
    "add invoice for Charlie Watts 1 Steady Beat Drum Lesson at 90 dollars",
    "create invoice customer Ron Wood 1 Faces Guitar Collection at 45 dollars",
    "make invoice for Jimmy Page 1 Led Zeppelin Guitar Tab Book at 35 dollars",
    "new invoice customer Robert Plant 1 Stairway to Heaven Vocal Score at 25 dollars",
    "generate invoice for John Paul Jones 1 Bass and Keys Lesson at 150 dollars",
    "add invoice customer John Bonham 1 When the Levee Breaks Drum Chart at 40 dollars",
    "create invoice for Roger Waters 1 Pink Floyd Bass Collection at 50 dollars",
    "make invoice customer David Gilmour 1 Comfortably Numb Guitar Solo Tab at 20 dollars",
    "new invoice for Nick Mason 1 Pink Floyd Drum Book at 32 dollars",
    "generate invoice customer Richard Wright 1 Great Gig in Sky Keyboard Score at 28 dollars",
    "add invoice for Syd Barrett 1 Diamond Dogs Guitar Songbook at 24 dollars",
    "create invoice for customer ABC Corp with 100 Business Cards at 2 dollars each",
    "make invoice customer XYZ Ltd 500 Letterheads at 1 dollar per piece",
    "new invoice for DEF Inc 1000 Envelopes at 0.50 each",
    "generate invoice customer GHI Co 50 Folders at 3 dollars per item",
    "add invoice for JKL Corp 200 Binders at 12 dollars each",
    "create invoice customer MNO Ltd 25 Calculators at 45 dollars per unit",
    "make invoice for PQR Inc 10 Staplers at 18 dollars each",
    "new invoice customer STU Co 300 Pens at 2 dollars per piece",
    "generate invoice for VWX Corp 150 Notebooks at 8 dollars each",
    "add invoice customer YZA Ltd 75 Rulers at 3 dollars per item",
    "create invoice for customer 123 Corp with 20 Whiteboards at 45 dollars each",
    "make invoice customer 456 Ltd 5 Projectors at 300 dollars per unit",
    "new invoice for 789 Inc 30 Presentation Remotes at 40 dollars each",
    "generate invoice customer 101 Co 12 Flipchart Pads at 15 dollars per item",
    "add invoice for 202 Corp 8 Easels at 30 dollars each",
    "create invoice customer 303 Ltd 40 Markers at 4 dollars per piece",
    "make invoice for 404 Inc 60 Highlighters at 6 dollars each",
    "new invoice customer 505 Co 90 Erasers at 1 dollar per item",
    "generate invoice for 606 Corp 15 Scissors at 10 dollars each",
    "add invoice customer 707 Ltd 25 Glue Sticks at 3 dollars per piece",
]

# Create labels for each training example
labels = []

# view_invoice: first 150 examples
labels.extend(['view_invoice'] * 150)

# list_invoices: next 150 examples  
labels.extend(['list_invoices'] * 150)

# create_invoice: next 50 examples
labels.extend(['create_invoice'] * 50)

# edit_invoice: next 50 examples
labels.extend(['edit_invoice'] * 50)

# list_customers: next 50 examples
labels.extend(['list_customers'] * 50)

# list_products: next 50 examples
labels.extend(['list_products'] * 50)

# show_reports: next 50 examples
labels.extend(['show_reports'] * 50)

# overdue_invoices: next 50 examples
labels.extend(['overdue_invoices'] * 50)

# help: next 50 examples
labels.extend(['help'] * 50)

# create_product_with_data: next 150 examples
labels.extend(['create_product_with_data'] * 150)

# create_customer_with_data: next 150 examples
labels.extend(['create_customer_with_data'] * 150)

# create_invoice_with_data: next 150 examples
labels.extend(['create_invoice_with_data'] * 150)

print(f"Total training examples: {len(training_data)}")
print(f"Total labels: {len(labels)}")
print(f"Label distribution:")
for label in set(labels):
    count = labels.count(label)
    print(f"  {label}: {count} examples")

In [ ]:
# Create dataset and label mappings
df = pd.DataFrame({
    'text': training_data,
    'label': labels
})

# Create label mappings for 12 actions
unique_labels = sorted(df['label'].unique())
label2id = {label: idx for idx, label in enumerate(unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}

# Convert string labels to numeric IDs
df['label_id'] = df['label'].map(label2id)

print(f"\nLabel mappings for 12-action model:")
for label, id_val in label2id.items():
    print(f"  {id_val}: {label}")

# Save label mappings for integration
class_mappings = {
    "action_classes": label2id,
    "label_to_action": id2label
}

with open('class_mappings_12_actions.json', 'w') as f:
    json.dump(class_mappings, f, indent=2)

print(f"\nDataset shape: {df.shape}")
print(df.head())

In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    df['text'].tolist(), 
    df['label_id'].tolist(), 
    test_size=0.2, 
    random_state=42,
    stratify=df['label_id'].tolist()  # Ensure balanced split
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

# Verify label distribution in splits
train_label_counts = pd.Series(y_train).value_counts().sort_index()
test_label_counts = pd.Series(y_test).value_counts().sort_index()

print("\nTraining set label distribution:")
for idx, count in train_label_counts.items():
    print(f"  {id2label[idx]}: {count} examples")

print("\nTest set label distribution:")
for idx, count in test_label_counts.items():
    print(f"  {id2label[idx]}: {count} examples")

In [ ]:
# Initialize tokenizer and model for 12 classes
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

# Initialize model with 12 labels
model = DistilBertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(unique_labels),
    id2label=id2label,
    label2id=label2id
)

print(f"Model initialized with {len(unique_labels)} classes")
print(f"Model config: {model.config}")

In [ ]:
# Tokenize datasets
def tokenize_function(examples):
    return tokenizer(
        examples['text'], 
        truncation=True, 
        padding=True, 
        max_length=128
    )

# Create train and test datasets
train_dataset = Dataset.from_dict({
    'text': X_train,
    'labels': y_train
})

test_dataset = Dataset.from_dict({
    'text': X_test,
    'labels': y_test
})

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

print(f"Train dataset: {train_dataset}")
print(f"Test dataset: {test_dataset}")

In [ ]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results_12_actions',
    num_train_epochs=5,  # Increased epochs for better convergence
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs_12_actions',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    save_total_limit=2,
    dataloader_num_workers=2,
    learning_rate=2e-5,
)

# Define metrics
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    accuracy = accuracy_score(labels, predictions)
    return {'accuracy': accuracy}

# Initialize trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print("Trainer initialized for 12-action classification")

In [ ]:
# Train the model
print("🚀 Starting training for enhanced 12-action model...")
trainer.train()
print("✅ Training completed!")

In [ ]:
# Evaluate the model
print("📊 Evaluating model performance...")
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

# Get predictions for detailed analysis
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

# Print classification report
print("\n📈 Detailed Classification Report:")
print(classification_report(
    y_test, y_pred, 
    target_names=[id2label[i] for i in range(len(unique_labels))],
    digits=4
))

# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=[id2label[i] for i in range(len(unique_labels))],
    yticklabels=[id2label[i] for i in range(len(unique_labels))]
)
plt.title('Enhanced 12-Action Invoice Classifier - Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Calculate per-class accuracy
per_class_accuracy = cm.diagonal() / cm.sum(axis=1)
print("\n🎯 Per-class Accuracy:")
for i, accuracy in enumerate(per_class_accuracy):
    print(f"  {id2label[i]}: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
# Test model with sample entity creation queries
print("🧪 Testing Enhanced Model with Entity Creation Queries")
print("=" * 60)

test_queries = [
    # Original actions
    "view invoice 123",
    "list all invoices", 
    "create new invoice",
    "show customers",
    "help me",
    
    # New entity creation actions
    "create product WaterBottle price 5 dollars",
    "add customer John Smith email john@example.com phone 555-1234",
    "create invoice for customer Sarah with 2 laptops at 1000 each",
    "make product Coffee Mug priced at 15 USD",
    "register customer Mike Wilson address 123 Main St",
    "new invoice for David with 5 water bottles at 5 dollars each"
]

def test_classification(query, model, tokenizer, id2label):
    inputs = tokenizer(query, return_tensors='pt', truncation=True, padding=True, max_length=128)
    
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)
        predicted_class_id = predictions.argmax().item()
        confidence = predictions[0][predicted_class_id].item()
    
    return id2label[predicted_class_id], confidence

for query in test_queries:
    predicted_action, confidence = test_classification(query, model, tokenizer, id2label)
    confidence_icon = "🎯" if confidence > 0.8 else "🔍" if confidence > 0.6 else "❓"
    print(f"{confidence_icon} Query: \"{query}\"")
    print(f"    → Action: {predicted_action}")
    print(f"    → Confidence: {confidence:.4f} ({confidence*100:.2f}%)")
    print()

## Model Export for Browser Integration

Export the trained model to ONNX format for use with transformers.js in the browser.

In [ ]:
# Save the fine-tuned model and tokenizer
model_save_path = './enhanced_invoice_classifier_12_actions'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"✅ Model and tokenizer saved to {model_save_path}")

# Save updated configuration with 12 labels
config_data = {
    "activation": "gelu",
    "architectures": ["DistilBertForSequenceClassification"],
    "attention_dropout": 0.1,
    "dim": 768,
    "dropout": 0.1,
    "hidden_dim": 3072,
    "id2label": id2label,
    "initializer_range": 0.02,
    "label2id": label2id,
    "max_position_embeddings": 512,
    "model_type": "distilbert",
    "n_heads": 12,
    "n_layers": 6,
    "pad_token_id": 0,
    "problem_type": "single_label_classification",
    "qa_dropout": 0.1,
    "seq_classif_dropout": 0.2,
    "sinusoidal_pos_embds": False,
    "tie_weights_": True,
    "torch_dtype": "float32",
    "transformers_version": "4.53.3",
    "vocab_size": 30522
}

with open(f'{model_save_path}/config.json', 'w') as f:
    json.dump(config_data, f, indent=2)

print("✅ Enhanced config.json saved with 12 action classes")

In [ ]:
# Convert to ONNX for browser deployment
print("🔄 Converting model to ONNX format for browser deployment...")

try:
    # Load the model for ONNX conversion
    ort_model = ORTModelForSequenceClassification.from_pretrained(
        model_save_path,
        export=True,
        provider="CPUExecutionProvider"
    )
    
    # Save ONNX model
    onnx_save_path = './enhanced_invoice_classifier_12_actions_onnx'
    ort_model.save_pretrained(onnx_save_path)
    tokenizer.save_pretrained(onnx_save_path)
    
    # Copy config and class mappings to ONNX directory
    import shutil
    shutil.copy(f'{model_save_path}/config.json', f'{onnx_save_path}/config.json')
    shutil.copy('class_mappings_12_actions.json', f'{onnx_save_path}/class_mappings.json')
    
    print(f"✅ ONNX model saved to {onnx_save_path}")
    print("📦 Ready for browser integration!")
    
    # List all files in ONNX directory
    import os
    print(f"\n📁 Files in {onnx_save_path}:")
    for file in os.listdir(onnx_save_path):
        file_path = os.path.join(onnx_save_path, file)
        file_size = os.path.getsize(file_path) / (1024*1024)  # MB
        print(f"  📄 {file} ({file_size:.2f} MB)")
    
except Exception as e:
    print(f"❌ ONNX conversion failed: {e}")
    print("💡 You can still use the PyTorch model or convert manually")

In [ ]:
# Create integration instructions
integration_guide = """
# Enhanced Invoice Classifier Integration Guide

## Model Summary
- **Model Type**: DistilBERT for Sequence Classification
- **Actions**: 12 total (9 existing + 3 new entity creation)
- **Training Examples**: 1200+ total
- **Accuracy**: {accuracy:.2f}%

## New Action Classes
10. **create_product_with_data** - Extract product data and pre-fill form
11. **create_customer_with_data** - Extract customer data and pre-fill form  
12. **create_invoice_with_data** - Extract invoice data and pre-fill form

## Integration Steps

### 1. Copy Model Files
Copy all files from `enhanced_invoice_classifier_12_actions_onnx/` to:
`/public/models/invoice-classifier/`

### 2. Update InvoiceClassifier.js
- Update action mappings to include 12 actions
- Add new action types for entity creation
- Handle data extraction results

### 3. Create EntityExtractor.js
- Implement natural language data parsing
- Extract structured data from commands
- Validate extracted information

### 4. Enhance Form Components
- ProductPage: Accept pre-filled product data
- CustomerPage: Accept pre-filled customer data
- InvoicePage: Accept pre-filled invoice data

### 5. Update LLMAssistant.jsx
- Handle new action types
- Call EntityExtractor for data parsing
- Route with extracted data

## Example Commands
- "create product WaterBottle price 5 dollars" → extract + route to ProductPage
- "add customer John Smith email john@example.com" → extract + route to CustomerPage
- "create invoice for Sarah with 2 laptops at 1000 each" → extract + route to InvoicePage

## Files to Deploy
- model.onnx ({model_size:.1f} MB)
- config.json
- tokenizer.json
- tokenizer_config.json
- special_tokens_map.json
- vocab.txt
- class_mappings.json
"""

# Get model file size
try:
    model_size = os.path.getsize(f'{onnx_save_path}/model.onnx') / (1024*1024)
    accuracy = eval_results['eval_accuracy'] * 100
except:
    model_size = 0
    accuracy = 0

formatted_guide = integration_guide.format(
    accuracy=accuracy,
    model_size=model_size
)

# Save integration guide
with open('Enhanced_Integration_Guide.md', 'w') as f:
    f.write(formatted_guide)

print("📋 Integration guide saved as 'Enhanced_Integration_Guide.md'")
print("\n🎉 Enhanced 12-Action Invoice Classifier Training Complete!")
print("\n🚀 Next Steps:")
print("1. Copy ONNX model files to your React app's /public/models/invoice-classifier/")
print("2. Update InvoiceClassifier.js with 12-action support")
print("3. Create EntityExtractor.js service")
print("4. Enhance form components for pre-filling")
print("5. Update LLMAssistant.jsx routing logic")
print("\n💡 The model now supports intelligent entity creation with natural language!")